# ③' 模範グループワーク例（完成サンプル）
**岡山県 高校教員向け データサイエンス・ハンズオン（3時間目の見本）**

グループワークの「完成イメージ」です。手が止まったチームは、この流れをまねしてください。

> 発表の型：**問い → 使うデータ → 分析 → わかったこと → 注意・限界**


## 0. 準備

In [ ]:
# === データ読み込みの準備（このセルを最初に実行）===
import pandas as pd, io, os
BASE_URL = "https://raw.githubusercontent.com/yasuyuki-nogami/ds-handson/main/data/"

def load(name):
    if os.path.exists("data/" + name):
        return pd.read_csv("data/" + name)
    if BASE_URL:
        try:
            return pd.read_csv(BASE_URL + name)
        except Exception:
            pass
    from google.colab import files
    print(name, " をアップロードしてください")
    up = files.upload(); key = list(up.keys())[0]
    return pd.read_csv(io.BytesIO(up[key]))
print("準備OK")


In [ ]:
!pip -q install japanize-matplotlib
import matplotlib.pyplot as plt
import japanize_matplotlib
print("日本語フォント準備OK")


---
# 例A：「スマホ時間が長いほど、点数は低い？」
**使うデータ**：`class_sample.csv`（生徒360人の練習用・架空データ）


In [ ]:
df = load("class_sample.csv")
df.head()


### 分析1：まず散布図で全体を見る

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(df["smartphone_hours"], df["test_score"], alpha=0.4)
plt.xlabel("スマホ利用時間（時間/日）"); plt.ylabel("テストの点数")
plt.title("スマホ時間と点数")
plt.show()

print("相関係数：", round(df["smartphone_hours"].corr(df["test_score"]), 3))


### 分析2：回帰直線であてはめる

In [ ]:
import numpy as np
x = df["smartphone_hours"]; y = df["test_score"]
a, b = np.polyfit(x, y, 1)
print(f"傾き = {a:.2f} → スマホが1時間増えると、点数は約 {a:.1f} 点変化（このデータ上では）")

plt.figure(figsize=(6,5))
plt.scatter(x, y, alpha=0.3)
xx = np.linspace(x.min(), x.max(), 100)
plt.plot(xx, a*xx + b, color="red", linewidth=2)
plt.xlabel("スマホ利用時間（時間/日）"); plt.ylabel("点数"); plt.title("スマホ時間 → 点数（単回帰）")
plt.show()


### 分析3：群に分けて平均を比べる

In [ ]:
# スマホ時間を「2時間未満 / 2時間以上」で分けて平均点を比較
df["スマホ区分"] = np.where(df["smartphone_hours"] >= 2, "2時間以上", "2時間未満")
print(df.groupby("スマホ区分")["test_score"].mean().round(1))

df.boxplot(column="test_score", by="スマホ区分", figsize=(6,4))
plt.title("スマホ時間の区分と点数"); plt.suptitle(""); plt.ylabel("点数")
plt.show()


### 発表まとめ（記入例）
```
問い：スマホ時間が長いほど点数は低い？
使ったデータ：class_sample.csv（生徒360人・架空データ）
分析：散布図・相関係数・単回帰・群間比較
わかったこと：弱い負の相関。スマホ時間が長い群ほど平均点がやや低い。
注意・限界：
  ・架空データなので現実の値そのものではない。
  ・相関があっても「スマホが原因」とは言えない（相関≠因果）。
    生活リズム全体や勉強時間など別の要因（交絡）が隠れている可能性。
```


---
# 例B：「人口密度が高い地域と低い地域で、面積はどう違う？」
**使うデータ**：`prefecture.csv`（47都道府県・実データ）


In [ ]:
pref = load("prefecture.csv")

# 人口密度の中央値で「高い群 / 低い群」に分ける
med = pref["density_per_km2"].median()
pref["密度区分"] = np.where(pref["density_per_km2"] >= med, "高い", "低い")
print(pref.groupby("密度区分")[["area_km2","population_2020"]].mean().round(0))


In [ ]:
# 地方別の平均人口密度
g = pref.groupby("region")["density_per_km2"].mean().sort_values()
plt.figure(figsize=(7,4))
plt.barh(g.index, g.values)
plt.title("地方別の平均人口密度"); plt.xlabel("人口密度（人/km²）")
plt.tight_layout(); plt.show()


### 発表まとめ（記入例）
```
問い：人口密度が高い地域と低い地域で面積はどう違う？
使ったデータ：prefecture.csv（国勢調査2020ほか・実データ）
分析：中央値で2群に分けて平均を比較、地方別の平均密度を棒グラフ化
わかったこと：人口密度が低い群は平均面積が大きい傾向（広い県ほど密度が低い）。
        関東・近畿は平均密度が高い。
注意・限界：都道府県は47件と少ない。市町村まで見ると別の傾向が出るかも。
```


---
## あなたのチームでやってみよう
上の例をまねて、別のテーマ（睡眠×点数、朝食×点数、身長×体重 など）でも試してみましょう。
発表スライドは `slide_prompt.txt` の指示文をAIに渡すと自動生成できます。